<a href="https://colab.research.google.com/github/Cyb3rVigil/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cyb3rVigil/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip -q install -U duckdb pandas scikit-learn pyarrow huggingface_hub

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN is not None, "Add HF_TOKEN in Colab Secrets."

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"{BASE}/fact_content_daily_performance/month=2026-03/*.parquet"
FACT_APR = f"{BASE}/fact_content_daily_performance/month=2026-04/*.parquet"

In [ ]:
con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{FACT_MAR}')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


# W03 Feature Leakage Check

Lane: Content refresh / quick-win ranking.

Goal: Build an honest March feature frame, create an April outcome label, then intentionally add leaked columns to prove why leakage creates fake model performance.

Decision moment: End of March 2026.

Safe feature window: March 2026 only.

Label/proxy window: April 2026.

I will not use June 2026 `_sample` for label development because it is the final month and acts like a sealed test window.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW march_features AS
SELECT
  client_hash_id,
  content_hash_id,

  COUNT(DISTINCT report_date) AS active_days,
  SUM(gsc_clicks) AS march_clicks,
  SUM(gsc_impressions) AS march_impressions,
  100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS march_ctr_pct,

  SUM(NULLIF(gsc_avg_position, 0) * gsc_impressions)
    / NULLIF(SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_impressions ELSE 0 END), 0)
    AS march_avg_position

FROM read_parquet('{FACT_MAR}')
GROUP BY 1, 2
HAVING SUM(gsc_impressions) >= 10
""")

con.sql("""
SELECT *
FROM march_features
ORDER BY march_impressions DESC
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,active_days,march_clicks,march_impressions,march_ctr_pct,march_avg_position
0,client_e547b89c05043229,content_eadb33b5df496f4a,31,5668.0,617124.0,0.918454,2.331470
1,client_e547b89c05043229,content_ec2e0346994fb5a5,31,1480.0,245276.0,0.603402,2.757730
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,31,669.0,244931.0,0.273138,15.173490
3,client_e547b89c05043229,content_0e03de7680314cd5,31,720.0,221310.0,0.325336,2.506100
4,client_23a62021009f63c4,content_44f34c0a90047651,31,24.0,212404.0,0.011299,0.665877
5,client_62f4a7e64f5e0096,content_7172a7fad43f0998,31,862.0,205867.0,0.418717,3.298139
6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,31,2446.0,205045.0,1.192909,4.453174
7,client_e547b89c05043229,content_8d7d99f109e19aa2,31,289.0,203497.0,0.142017,2.468557
8,client_62f4a7e64f5e0096,content_f107e54b10b43725,31,996.0,195997.0,0.508171,3.179023
9,client_23a62021009f63c4,content_36e53e9c707674fc,31,242.0,194579.0,0.124371,32.786981


### Feature vector plan

One row means one pseudonymized content item.

I build a small feature vector for a refresh / CTR-fix lane. The target-like or future-derived fields are excluded from the feature list. IDs are kept only as context.

The final feature vector uses search visibility, CTR, average position, freshness, content length, and basic categorical content type handling.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Honest Feature Frame

All five features are safe because they are computed only from March 2026 data, which is available at the March decision moment.

- `active_days`: count of March days with available GSC data.
- `march_clicks`: March clicks only.
- `march_impressions`: March impressions only.
- `march_ctr_pct`: March clicks divided by March impressions.
- `march_avg_position`: March average search position weighted by impressions.

IDs are kept only as context/join keys, not model features.

### Feature notes

Each feature below is available before prediction because it comes from current/past observed page performance or content metadata. I do not use label-derived trend fields.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage hunt

I test leakage by comparing an honest model using only March features against models that intentionally use April/future-derived columns.

Blocked fields are:

- `april_clicks`
- `april_impressions`
- `april_ctr_pct`
- `LEAK_april_clicks`
- `LEAK_april_ctr_pct`
- `LEAK_label_copy`

In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW april_outcomes AS
SELECT
  client_hash_id,
  content_hash_id,
  SUM(gsc_clicks) AS april_clicks,
  SUM(gsc_impressions) AS april_impressions,
  100.0 * SUM(gsc_clicks)
    / NULLIF(SUM(gsc_impressions), 0) AS april_ctr_pct
FROM read_parquet('{FACT_APR}')
GROUP BY 1, 2
""")

df = con.sql("""
SELECT
  f.*,
  a.april_clicks,
  a.april_impressions,
  a.april_ctr_pct
FROM march_features AS f
INNER JOIN april_outcomes AS a
USING (client_hash_id, content_hash_id)
WHERE a.april_clicks IS NOT NULL
""").df()

df = df.dropna().copy()
df = df.sample(n=min(len(df), 50000), random_state=42)

df["label_top_april_clicks"] = (
    df["april_clicks"] >= df["april_clicks"].quantile(0.75)
).astype(int)

print("Rows:", len(df))
print(df["label_top_april_clicks"].value_counts())
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 50000
label_top_april_clicks
0    35091
1    14909
Name: count, dtype: int64


,client_hash_id,content_hash_id,active_days,march_clicks,march_impressions,march_ctr_pct,march_avg_position,april_clicks,april_impressions,april_ctr_pct,label_top_april_clicks
80591,client_23a62021009f63c4,content_f1cf45fcc572366f,31,1.0,416.0,0.240385,22.781553,0.0,440.0,0.000000,0
79765,client_08a6a72ff48e62c0,content_6af4987f1f19f009,31,1.0,113.0,0.884956,26.566372,1.0,23.0,4.347826,0
50478,client_20259bd6705d81d4,content_7a3bc893e1177236,31,5.0,5706.0,0.087627,2.795478,9.0,4220.0,0.213270,1
50783,client_20259bd6705d81d4,content_763c9885ffdf8596,31,0.0,642.0,0.000000,2.277165,0.0,262.0,0.000000,0
84368,client_157ffe4d4a595515,content_c6975cd15739337f,31,0.0,15.0,0.000000,5.400000,0.0,25.0,0.000000,0


## Label

Label: `label_top_april_clicks`

Meaning: 1 if the page is in the top 25% of April clicks among the sampled March-observed pages.

This label is future information from the March decision moment. It is allowed as the target, but April-derived columns must never become features.

In [ ]:
honest_features = [
    "active_days",
    "march_clicks",
    "march_impressions",
    "march_ctr_pct",
    "march_avg_position",
]

def score_model(feature_cols):
    X = df[feature_cols]
    y = df["label_top_april_clicks"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.30,
        random_state=42,
        stratify=y
    )

    clf = DecisionTreeClassifier(max_depth=3, random_state=42)
    clf.fit(X_train, y_train)

    pred = clf.predict_proba(X_test)[:, 1]

    return {
        "features_used": ", ".join(feature_cols),
        "roc_auc": roc_auc_score(y_test, pred),
        "average_precision": average_precision_score(y_test, pred),
    }

honest_score = score_model(honest_features)
pd.DataFrame([honest_score])

,features_used,roc_auc,average_precision
0,"active_days, march_clicks, march_impressions, ...",0.911335,0.843425


In [ ]:
df["LEAK_april_clicks"] = df["april_clicks"]
df["LEAK_april_ctr_pct"] = df["april_ctr_pct"]
df["LEAK_label_copy"] = df["label_top_april_clicks"]

leak_tests = [
    {
        "version": "honest_features_only",
        **score_model(honest_features)
    },
    {
        "version": "future_april_clicks_leak",
        **score_model(honest_features + ["LEAK_april_clicks"])
    },
    {
        "version": "future_april_ctr_leak",
        **score_model(honest_features + ["LEAK_april_ctr_pct"])
    },
    {
        "version": "direct_label_copy_leak",
        **score_model(honest_features + ["LEAK_label_copy"])
    },
]

leak_results = pd.DataFrame(leak_tests)
leak_results[["version", "roc_auc", "average_precision"]]

,version,roc_auc,average_precision
0,honest_features_only,0.911335,0.843425
1,future_april_clicks_leak,1.000000,1.000000
2,future_april_ctr_leak,0.968866,0.892796
3,direct_label_copy_leak,1.000000,1.000000


## Leakage Result

The honest model uses only March features, so its score is the real baseline.

The leaked versions perform better because they include information from April, the same future window used to create the label.

The worst leak is `LEAK_label_copy`, because it directly copies the answer into the feature set. A model using it is not learning; it is cheating.

Leakage rule learned: any column created from the label window, future outcome window, or the label itself must be excluded from the final feature frame.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields

I excluded label-derived fields, raw IDs, and product/model metadata that would not be appropriate as model features.

In [ ]:
final_feature_frame = df[
    ["client_hash_id", "content_hash_id"]
    + honest_features
    + ["label_top_april_clicks"]
].copy()

for bad_col in [
    "april_clicks",
    "april_impressions",
    "april_ctr_pct",
    "LEAK_april_clicks",
    "LEAK_april_ctr_pct",
    "LEAK_label_copy",
]:
    assert bad_col not in final_feature_frame.columns

print("Final frame shape:", final_feature_frame.shape)
print("Final columns:", final_feature_frame.columns.tolist())

final_feature_frame.head()

Final frame shape: (50000, 8)
Final columns: ['client_hash_id', 'content_hash_id', 'active_days', 'march_clicks', 'march_impressions', 'march_ctr_pct', 'march_avg_position', 'label_top_april_clicks']


,client_hash_id,content_hash_id,active_days,march_clicks,march_impressions,march_ctr_pct,march_avg_position,label_top_april_clicks
80591,client_23a62021009f63c4,content_f1cf45fcc572366f,31,1.0,416.0,0.240385,22.781553,0
79765,client_08a6a72ff48e62c0,content_6af4987f1f19f009,31,1.0,113.0,0.884956,26.566372,0
50478,client_20259bd6705d81d4,content_7a3bc893e1177236,31,5.0,5706.0,0.087627,2.795478,1
50783,client_20259bd6705d81d4,content_763c9885ffdf8596,31,0.0,642.0,0.000000,2.277165,0
84368,client_157ffe4d4a595515,content_c6975cd15739337f,31,0.0,15.0,0.000000,5.400000,0


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The feature vector is built with engineered features and categorical handling
- [x] Missing values are handled and explained
- [x] Leakage fields are tested and excluded
- [x] IDs are kept as context only, not model features
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.